# Cost Estimation and Optimization Lab

**Module:** M1 — LLM Fundamentals
**Lesson:** L4 — Tokens and Cost
**Audience:** AI Engineer (developer track)
**Format:** Homework — due before the next meeting
**Prereqs:** Lessons 1–3 completed, `docs/model-selection.md` from Lesson 3's homework, Python environment with Anthropic SDK

## Why this exercise

You're about to build a production AI system (your capstone). Before writing a line of application code, you need to know what it will cost. In this notebook you'll estimate your capstone's monthly LLM costs from first principles, then systematically reduce them with five optimization techniques—documenting the quality-cost tradeoff for each. By the end you'll also have a cost-monitoring logger that feeds directly into M6's cost engineering dashboard.

This is a direct continuation of Lesson 3's homework: you're costing out the exact tasks and models in your `docs/model-selection.md`. Do this after Lesson 3's notebook, not before — the task list comes from there. It costs nothing beyond the course's Claude key; there's no other provider involved in this lesson.

### Success criteria

You're done when:

- You have cost-calculation functions that compute per-request and monthly costs from token usage and pricing
- Your capstone has a cost estimate table: per-task cost, daily volume, monthly total
- You've applied at least 3 optimization techniques and documented the cost reduction *and* quality impact of each
- Your API wrapper logs token usage and cost per request to a CSV file
- Your quality checklist passes (see the end of this notebook)


## Setup

Run this cell once. It installs the dependencies and reads your API keys from Colab Secrets (or local env vars).

- In Colab: open the key icon in the left sidebar and add `ANTHROPIC_API_KEY`.
- Locally: `export ANTHROPIC_API_KEY=...` before launching Jupyter.

**Never paste an API key into a cell.** This notebook will be pushed to GitHub at the end of the course as portfolio evidence.

In [ ]:
%pip install -q anthropic

import os

try:
    from google.colab import userdata  # Colab
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY')

assert ANTHROPIC_API_KEY, 'Set ANTHROPIC_API_KEY in Colab Secrets or your shell env.'

# Model tier switch — see shared/cheaper-model-substitution.md
MODEL_TIER = os.environ.get('MODEL_TIER', 'cheap')
MODEL = {
    'cheap':    'claude-haiku-4-5',
    'standard': 'claude-sonnet-5',
    'premium':  'claude-opus-4-8',
    'supreme':  'claude-fable-5',
}[MODEL_TIER]
# Short family name matching the PRICING dict keys below — keeps cost lookups
# in sync with whichever MODEL_TIER is actually active.
MODEL_FAMILY = {
    'cheap':    'claude-haiku',
    'standard': 'claude-sonnet',
    'premium':  'claude-opus',
    'supreme':  'claude-fable',
}[MODEL_TIER]
print(f'Using model: {MODEL} (tier={MODEL_TIER})')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 15.9 MB/s eta 0:00:00
Using model: claude-haiku-4-5 (tier=cheap)


---

## Part 1 — Cost Calculator (~27 min)

## Step 1 — Pricing dict

Define per-token pricing for Claude Haiku, Sonnet, and Opus (input and output separately, per 1M tokens).

These prices are **volatile** — verify against https://www.anthropic.com/pricing before each cohort.

In [ ]:
# VOLATILE: Update per cohort from https://platform.claude.com/docs/en/about-claude/pricing
PRICING = {
    "claude-haiku": {          # Claude Haiku 4.5
        "input_per_1m": 1.00,
        "output_per_1m": 5.00,
    },
    "claude-sonnet": {         # Claude Sonnet 5 (standard price, no longer "introductory")
        "input_per_1m": 2.00,
        "output_per_1m": 10.00,
    },
    "claude-opus": {           # Claude Opus 5
        "input_per_1m": 5.00,
        "output_per_1m": 25.00,
    },
    "claude-fable": {          # Claude Fable 5
        "input_per_1m": 10.00,
        "output_per_1m": 50.00,
    },
}
PRICING

{'claude-haiku': {'input_per_1m': 1.0, 'output_per_1m': 5.0},
 'claude-sonnet': {'input_per_1m': 2.0, 'output_per_1m': 10.0},
 'claude-opus': {'input_per_1m': 5.0, 'output_per_1m': 25.0},
 'claude-fable': {'input_per_1m': 10.0, 'output_per_1m': 50.0}}

## Step 2 — Cost-calculation functions

Write two functions:

1. `calculate_cost(model, input_tokens, output_tokens)` → cost in USD for a single call
2. `estimate_monthly(model, input_tokens, output_tokens, requests_per_day)` → projected monthly cost

The formula: `cost = (input_tokens × input_price_per_token) + (output_tokens × output_price_per_token)`

In [ ]:
def calculate_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    """Return the cost in USD for a single API call."""
    rates = PRICING[model]
    input_cost = input_tokens * (rates["input_per_1m"] / 1_000_000)
    output_cost = output_tokens * (rates["output_per_1m"] / 1_000_000)
    return input_cost + output_cost


def estimate_monthly(model: str, input_tokens: int, output_tokens: int, requests_per_day: int) -> float:
    """Return projected monthly cost in USD."""
    per_request_cost = calculate_cost(model, input_tokens, output_tokens)
    return per_request_cost * requests_per_day * 30


# Sanity check: 1,000 input + 500 output tokens on Sonnet
cost = calculate_cost("claude-sonnet", 1_000, 500)
monthly = estimate_monthly("claude-sonnet", 1_000, 500, 100)
print(f"Per request: ${cost:.4f}")
print(f"Monthly (100 req/day): ${monthly:.2f}")

Per request: $0.0070
Monthly (100 req/day): $21.00


## Step 3 — Measure real token usage

Don't guess token counts — measure them. Send a representative prompt for one of your capstone tasks and read `response.usage.input_tokens` / `response.usage.output_tokens`.

If you don't have capstone tasks defined yet, use this sample prompt.

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

MODEL = "claude-sonnet-5"       # actual model string for the API call
MODEL_FAMILY = "claude-sonnet"  # key into your PRICING dict

# Representative Fathom transcript excerpt — swap in a real one from your capstone repo for a more accurate read
sample_transcript = """
[00:00:04] Sarah: Next item — we need to fix the disconnect between MailerLite and the
WordPress subscribers table. Right now if someone unsubscribes in MailerLite, WordPress
never finds out, so the subscribers table still shows them as active.
[00:00:22] Mike: Right, so we need a sync going both ways. When someone unsubscribes on
the MailerLite side, that needs to update the subscriber's status in the WordPress table
automatically — mark them unsubscribed, don't just delete the row.
[00:00:41] Sarah: Agreed. And separately — for the welcome email sequence, if someone
doesn't respond after a few days, let's not just keep emailing them. Let's follow up with
a text message instead, through Twilio.
[00:01:05] Mike: Makes sense. So the flow is: send the welcome email, wait for some kind
of response or engagement, and if there's no response within X days, trigger a Twilio SMS
follow-up instead.
[00:01:20] Sarah: And whatever they reply to the text — we should record that response in
a table in WordPress, so we have a record of who responded and what they said.
[00:01:34] Mike: And if someone responds to the SMS asking to be unsubscribed, that needs
to actually unsubscribe them — both in WordPress and back in MailerLite, so we don't keep
emailing or texting someone who opted out through the SMS channel.
[00:01:50] Sarah: Perfect. So to summarize: 1) two-way sync between MailerLite unsubscribes
and the WordPress subscriber table 2) Twilio SMS follow-up for non-responders to the
welcome email 3) log SMS responses to a WordPress table 4) SMS-based unsubscribe requests
propagate back to both WordPress and MailerLite.
"""

system_prompt = """You are a workflow extraction assistant. Given a meeting transcript,
extract structured automation requirements as JSON with these fields:
- triggers: list of events that should kick off automation
- actions: list of actions to take (email, slack, crm, etc.)
- action_items: list of concrete follow-up tasks with owners if mentioned
"""

response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    system=system_prompt,
    messages=[{"role": "user", "content": f"Extract workflow requirements from this transcript:\n\n{sample_transcript}"}],
)

print(f"Input tokens:  {response.usage.input_tokens}")
print(f"Output tokens: {response.usage.output_tokens}")
print(f"Cost:          ${calculate_cost(MODEL_FAMILY, response.usage.input_tokens, response.usage.output_tokens):.6f}")
# Pull out only the text blocks, ignore ThinkingBlock / other block types
text_blocks = [block.text for block in response.content if block.type == "text"]
answer_text = "\n".join(text_blocks)

print(f"\nResponse: {answer_text}")

Input tokens:  734
Output tokens: 300
Cost:          $0.004468

Response: ```json
{
  "triggers": [
    "Subscriber unsubscribes in MailerLite",
    "Welcome email sent to a new subscriber",
    "No response/engagement from subscriber within X days of welcome email",
    "Subscriber replies to Twilio SMS follow-up",
    "Subscriber requests unsubscribe via SMS reply"
  ],
  "actions": [
    "Update subscriber status to 'unsubscribed' in WordPress subscribers table when they unsubscribe in MailerLite (do not delete row)",
    "Send welcome email to new subscriber",
    "Monitor for response/engagement after welcome email",
    "Trigger Twilio SMS follow-up if no response within X days",
    "Log SMS reply content and responder info to a WordPress table",
    "If SMS reply requests unsubscribe: update status to


## Step 4 — Build your capstone cost table

For each AI task in your capstone (from your Lesson 3 model-selection document):

1. Measure average input tokens (send a sample prompt, read `usage`)
2. Measure average output tokens
3. Estimate daily request volume from your product requirements
4. Calculate monthly cost with `estimate_monthly()`

If you don't have capstone tasks yet, use the three generic tasks below.

In [ ]:
PRICING = {
    "claude-haiku":  {"input_per_1m": 1.00,  "output_per_1m": 5.00},
    "claude-sonnet": {"input_per_1m": 2.00,  "output_per_1m": 10.00},
    "claude-opus":   {"input_per_1m": 5.00,  "output_per_1m": 25.00},
    "claude-fable":  {"input_per_1m": 10.00, "output_per_1m": 50.00},
    "gemini-flash-lite": {"input_per_1m": 0.30, "output_per_1m": 2.50},
    "mistral-small":     {"input_per_1m": 0.15, "output_per_1m": 0.60},
}


def print_capstone_cost_report():
    # Tasks from model-selection.md — model assignment per Section 2, token counts from
    # measured/estimated values, req/day derived from Section 1's ~40 builds/month split
    # across each task's per-build frequency.
    tasks = [
        {"task": "Extraction",           "model": "claude-sonnet",     "in_tokens": 2500, "out_tokens": 800,  "req_per_day": 2},
        {"task": "Build Planning",       "model": "claude-opus",       "in_tokens": 3000, "out_tokens": 1500, "req_per_day": 2},
        {"task": "Payload Generation",   "model": "claude-sonnet",     "in_tokens": 1800, "out_tokens": 900,  "req_per_day": 8},
        {"task": "Error Recovery",       "model": "gemini-flash-lite", "in_tokens": 66,   "out_tokens": 78,   "req_per_day": 3},
        {"task": "Verification Summary", "model": "gemini-flash-lite", "in_tokens": 84,   "out_tokens": 95,   "req_per_day": 2},
    ]

    header = f"{'Task':<22}{'Model':<18}{'In Tokens':>10}{'Out Tokens':>12}{'Req/Day':>10}{'Monthly':>12}"
    print(header)
    print("-" * len(header))

    total_monthly = 0.0
    for t in tasks:
        monthly = estimate_monthly(t["model"], t["in_tokens"], t["out_tokens"], t["req_per_day"])
        total_monthly += monthly
        print(f"{t['task']:<22}{t['model']:<18}{t['in_tokens']:>10}{t['out_tokens']:>12}"
              f"{t['req_per_day']:>10}{'$' + f'{monthly:.4f}':>12}")

    print("-" * len(header))
    print(f"{'TOTAL':<22}{'':<18}{'':>10}{'':>12}{'':>10}{'$' + f'{total_monthly:.4f}':>12}")

    print(f"\nIs ${total_monthly:.2f}/month sustainable for your project?")


print_capstone_cost_report()
# TODO: print a header row with columns: Task, Model, In Tokens, Out Tokens, Req/Day, Monthly
# TODO: loop over tasks; call estimate_monthly() for each and accumulate a total
# TODO: print each row formatted; add a separator and totals row at the end
# TODO: print "Is $X/month sustainable for your project?" with the real total substituted in

Task                  Model              In Tokens  Out Tokens   Req/Day     Monthly
------------------------------------------------------------------------------------
Extraction            claude-sonnet           2500         800         2     $0.7800
Build Planning        claude-opus             3000        1500         2     $3.1500
Payload Generation    claude-sonnet           1800         900         8     $3.0240
Error Recovery        gemini-flash-lite         66          78         3     $0.0193
Verification Summary  gemini-flash-lite         84          95         2     $0.0158
------------------------------------------------------------------------------------
TOTAL                                                                        $6.9891

Is $6.99/month sustainable for your project?


**Explain it back:** What drives the cost most — the input size, the output size, the request volume, or the model choice? Write a one-sentence answer in the markdown cell below, naming the specific driver, not just "it's expensive."

**Before you answer, check yourself:** if every task in your table comes out driven by "volume" or "model choice," go back and check whether any task generates long-form output — summaries, code, reports. Output tokens are the most commonly underestimated variable. Re-measure that task specifically before you trust the total; don't guess it.


Answer:
Build Planning and Payload Generation cost the most, which is what I expected. Build Planning only has 2 req/day but it uses Opus, which is more expensive, and it's inputting a whole extraction from a Fathom transcript (a lot of tokens).

Payload Generation uses a cheaper model, sonnet, but takes an input of 1800 tokens (from the whole build plan generated in Build Planning) and also can have 8 tasks (which corresponds to all the different automations/ workflows that need to be built.

So this makes sense.


*Your answer:*



---

## Part 2 — Cost Optimization (~30 min)

Apply **at least 3** of the following 5 techniques to your cost estimate. For each technique, document:
- What you changed
- The cost impact (recalculated monthly cost)
- The quality impact (measured or reasoned about — not guessed)

## Step 5 — Technique 1: Prompt caching

Restructure one prompt so the system message is a stable prefix. Add `cache_control` to the system message. Call the API twice in quick succession and check for `cache_read_input_tokens` in the usage — if non-zero, caching is active.

**Quality impact:** None expected — the output is identical.

In [ ]:
build_planning_system_prompt = """You are the build-planning agent for a meeting-to-workflow automation pipeline. Given structured requirements extracted from a Fathom meeting transcript, you produce a dependency-ordered, conflict-free build plan describing exactly what must be created, in what order, across three target platforms: Make.com, MailerLite, and WordPress.

## Your responsibilities

1. Sequence build steps so nothing is referenced before it exists (e.g. a Make scenario must not reference a MailerLite list, WordPress webhook, or CRM field that hasn't been created yet in an earlier step).
2. Detect and flag conflicts between requirements before they reach the payload-generation stage. A conflict is any case where two requirements would create duplicate resources, contradictory automation logic, or reference an entity ambiguously (e.g. two different "welcome email" requirements that should be merged into one sequence rather than built twice).
3. Identify reuse opportunities. If an existing MailerLite list, WordPress custom post type, or Make scenario already satisfies part of a requirement, prefer reusing it over creating a duplicate, and say so explicitly in your output.
4. Produce a build plan that a downstream payload-generation step can execute step-by-step without needing to re-derive any of this reasoning.

## Platform-specific rules

**Make.com**
- Every scenario needs exactly one trigger module (webhook, polling, or scheduled) followed by one or more action modules.
- Branching logic (e.g. "notify Slack only if enterprise plan") must be modeled as a Router module with explicit filter conditions, not as conditional text inside a single action.
- Scenarios that write to MailerLite or WordPress must be sequenced after those targets exist.
- Prefer one scenario per logical workflow rather than splitting a single business process across multiple scenarios, unless the transcript explicitly describes separate independent triggers.

**MailerLite**
- Lists and segments must be created before any Make scenario references them.
- If a requirement implies a segment (e.g. "enterprise plan signups") rather than a full list, prefer a segment on an existing list over a new list, when one plausibly exists.
- Automation sequences (welcome series, day-3 follow-ups) belong to MailerLite's own automation builder, not Make, unless the transcript specifies external triggering logic Make must own.

**WordPress**
- Custom post types, ACF fields, and webhooks must be defined before any Make scenario or external integration depends on them.
- Contact form and webhook changes should be scoped to the specific plugin or theme component mentioned (e.g. Elementor Pro, Divi, ACF) rather than assumed to require a new plugin unless the transcript indicates otherwise.
- Flag any WordPress core version or plugin compatibility risk if the requirement touches a known legacy area.

## Conflict severity

Classify every detected conflict as one of: **blocking** (build cannot proceed until resolved — e.g. two contradictory trigger definitions for the same event), **advisory** (build can proceed, but a human should review — e.g. possible duplicate list), or **informational** (no action needed, noted for traceability).

## Output format

Respond with JSON only, structured as:
{
  "build_steps": [
    {
      "step_number": <int>,
      "platform": "make" | "mailerlite" | "wordpress",
      "description": "<what this step creates>",
      "depends_on": [<step_number>, ...],
      "reuses_existing": "<resource name>" | null
    }
  ],
  "conflicts": [
    {
      "severity": "blocking" | "advisory" | "informational",
      "description": "<what the conflict is>",
      "affected_steps": [<step_number>, ...]
    }
  ]
}

Do not include explanatory prose outside the JSON. Do not invent platform resources that were not implied by the requirements. If a requirement is genuinely ambiguous and cannot be resolved without more information, represent it as an advisory conflict rather than guessing at intent."""

# Extracted requirements — this is what feeds the build-planning step use answer_text from before

resp = client.messages.create(
    model="claude-opus-4-8",
    max_tokens=1500,
    system=[
        {
            "type": "text",
            "text": build_planning_system_prompt,  # the ~1000-token prompt from before
            "cache_control": {"type": "ephemeral"},
        }
    ],
    messages=[{"role": "user", "content": f"Extracted requirements:\n\n{answer_text}\n\nProduce the build plan."}],
)

cached = getattr(resp.usage, "cache_read_input_tokens", 0) or 0
created = getattr(resp.usage, "cache_creation_input_tokens", 0) or 0

print(f"input_tokens={resp.usage.input_tokens} output_tokens={resp.usage.output_tokens} "
      f"cache_created={created} cache_read={cached}")
print(resp.content[0].text)


input_tokens=309 output_tokens=1467 cache_created=1337 cache_read=0
{
  "build_steps": [
    {
      "step_number": 1,
      "platform": "wordpress",
      "description": "Ensure/create the WordPress subscribers custom table (or CPT) with fields including subscriber identity, status (subscribed/unsubscribed), and reference key. Status updates must overwrite existing rows, not delete them.",
      "depends_on": [],
      "reuses_existing": "WordPress subscribers table"
    },
    {
      "step_number": 2,
      "platform": "wordpress",
      "description": "Create/extend a WordPress table (or fields on existing table) to log Twilio SMS reply content, responder info, and timestamp.",
      "depends_on": [],
      "reuses_existing": null
    },
    {
      "step_number": 3,
      "platform": "wordpress",
      "description": "Expose a WordPress webhook/REST endpoint for Make.com to write subscriber status updates and SMS reply logs into the tables from steps 1 and 2.",
      "depends_on":

Results: the first time I ran this function I got cache_created=1337 cache_read=0

The second time I ran this, I got cache_created=0 cache_read=1337

Which means that the second time it was reading from the cache.

## Step 6 — Technique 2: Prompt compression

Take your longest system prompt. Create a compressed version — aim for 30% fewer tokens. Run both versions on 3 test inputs and compare outputs.

**Quality impact:** Varies. If outputs are equivalent, the original had unnecessary padding.

In [ ]:
# ── Run this entire cell together, top to bottom, no earlier reruns in between ──
import anthropic
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

build_planning_original_prompt = """You are the build-planning agent for a meeting-to-workflow automation pipeline. Given structured requirements extracted from a Fathom meeting transcript, you produce a dependency-ordered, conflict-free build plan describing exactly what must be created, in what order, across three target platforms: Make.com, MailerLite, and WordPress.

## Your responsibilities

1. Sequence build steps so nothing is referenced before it exists (e.g. a Make scenario must not reference a MailerLite list, WordPress webhook, or CRM field that hasn't been created yet in an earlier step).
2. Detect and flag conflicts between requirements before they reach the payload-generation stage. A conflict is any case where two requirements would create duplicate resources, contradictory automation logic, or reference an entity ambiguously (e.g. two different "welcome email" requirements that should be merged into one sequence rather than built twice).
3. Identify reuse opportunities. If an existing MailerLite list, WordPress custom post type, or Make scenario already satisfies part of a requirement, prefer reusing it over creating a duplicate, and say so explicitly in your output.
4. Produce a build plan that a downstream payload-generation step can execute step-by-step without needing to re-derive any of this reasoning.

## Platform-specific rules

**Make.com**
- Every scenario needs exactly one trigger module (webhook, polling, or scheduled) followed by one or more action modules.
- Branching logic (e.g. "notify Slack only if enterprise plan") must be modeled as a Router module with explicit filter conditions, not as conditional text inside a single action.
- Scenarios that write to MailerLite or WordPress must be sequenced after those targets exist.
- Prefer one scenario per logical workflow rather than splitting a single business process across multiple scenarios, unless the transcript explicitly describes separate independent triggers.

**MailerLite**
- Lists and segments must be created before any Make scenario references them.
- If a requirement implies a segment (e.g. "enterprise plan signups") rather than a full list, prefer a segment on an existing list over a new list, when one plausibly exists.
- Automation sequences (welcome series, day-3 follow-ups) belong to MailerLite's own automation builder, not Make, unless the transcript specifies external triggering logic Make must own.

**WordPress**
- Custom post types, ACF fields, and webhooks must be defined before any Make scenario or external integration depends on them.
- Contact form and webhook changes should be scoped to the specific plugin or theme component mentioned (e.g. Elementor Pro, Divi, ACF) rather than assumed to require a new plugin unless the transcript indicates otherwise.
- Flag any WordPress core version or plugin compatibility risk if the requirement touches a known legacy area.

## Conflict severity

Classify every detected conflict as one of: **blocking** (build cannot proceed until resolved — e.g. two contradictory trigger definitions for the same event), **advisory** (build can proceed, but a human should review — e.g. possible duplicate list), or **informational** (no action needed, noted for traceability).

## Output format

Respond with JSON only, structured as:
{
  "build_steps": [
    {
      "step_number": <int>,
      "platform": "make" | "mailerlite" | "wordpress",
      "description": "<what this step creates>",
      "depends_on": [<step_number>, ...],
      "reuses_existing": "<resource name>" | null
    }
  ],
  "conflicts": [
    {
      "severity": "blocking" | "advisory" | "informational",
      "description": "<what the conflict is>",
      "affected_steps": [<step_number>, ...]
    }
  ]
}

Do not include explanatory prose outside the JSON. Do not invent platform resources that were not implied by the requirements. If a requirement is genuinely ambiguous and cannot be resolved without more information, represent it as an advisory conflict rather than guessing at intent."""

build_planning_compressed_prompt = """You are the build-planning agent for a meeting-to-workflow automation pipeline. Given structured requirements extracted from a Fathom transcript, produce a dependency-ordered, conflict-free build plan across Make.com, MailerLite, and WordPress.

Responsibilities:
1. Sequence steps so nothing is referenced before it exists (a Make scenario must not reference a MailerLite list, WordPress webhook, or CRM field that doesn't exist yet).
2. Detect conflicts: duplicate resources, contradictory automation logic, or ambiguous references (e.g. two "welcome email" requirements that should merge into one sequence, not be built twice).
3. Prefer reusing an existing MailerLite list, WordPress post type, or Make scenario over creating a duplicate; state the reuse explicitly.
4. Output steps that payload-generation can execute without re-deriving this reasoning.

Platform rules:
- **Make.com**: one trigger module + 1+ action modules per scenario. Conditional logic ("notify Slack only if enterprise") must be a Router module with explicit filters, not conditional text in one action. Sequence after MailerLite/WordPress targets exist. One scenario per logical workflow unless the transcript describes separate independent triggers.
- **MailerLite**: lists/segments created before any Make scenario references them. Prefer a segment on an existing list over a new list when one plausibly exists. Automation sequences (welcome series, day-3 follow-ups) belong to MailerLite's own builder, not Make, unless external triggering logic is specified.
- **WordPress**: custom post types, ACF fields, and webhooks must be defined before any dependent step. Scope changes to the specific plugin/theme mentioned (Elementor Pro, Divi, ACF) rather than assuming a new plugin. Flag WordPress core or plugin compatibility risk on known legacy areas.

Conflict severity: **blocking** (build cannot proceed — e.g. contradictory trigger definitions for the same event), **advisory** (build proceeds, human should review — e.g. possible duplicate list), **informational** (no action needed, logged for traceability).

Output JSON only, structured as:
{
  "build_steps": [
    {"step_number": <int>, "platform": "make"|"mailerlite"|"wordpress", "description": "<what this step creates>", "depends_on": [<step_number>, ...], "reuses_existing": "<resource name>"|null}
  ],
  "conflicts": [
    {"severity": "blocking"|"advisory"|"informational", "description": "<what the conflict is>", "affected_steps": [<step_number>, ...]}
  ]
}

No prose outside the JSON. Do not invent platform resources not implied by the requirements. Represent genuine ambiguity as an advisory conflict rather than guessing at intent."""

# Sanity check BEFORE calling the API — catches stale-variable bugs immediately
orig_len, comp_len = len(build_planning_original_prompt), len(build_planning_compressed_prompt)
print(f"original: {orig_len} chars (~{orig_len//4} tokens)")
print(f"compressed: {comp_len} chars (~{comp_len//4} tokens)")
assert orig_len > comp_len * 1.2, f"original ({orig_len} chars) isn't meaningfully longer than compressed ({comp_len} chars) — check for a stale/reused variable name!"

test_inputs = [
    """{"triggers": ["new signup webhook", "day-3 profile incomplete check"], "actions": ["send welcome email", "send day-3 follow-up email", "Slack notify sales team on enterprise signup", "log to CRM"], "action_items": ["build welcome email sequence", "build day-3 follow-up", "Slack webhook for enterprise signups", "CRM logging via Make scenario"]}""",
    """{"triggers": ["MailerLite unsubscribe event", "welcome email sent with no engagement after X days", "inbound Twilio SMS reply", "SMS reply contains unsubscribe request"], "actions": ["update WordPress subscribers table status to unsubscribed on MailerLite unsubscribe", "send Twilio SMS follow-up to non-responders", "log SMS response to WordPress table", "unsubscribe contact in WordPress and MailerLite on SMS unsubscribe request"], "action_items": ["build MailerLite -> WordPress unsubscribe sync (webhook or polling)", "build non-response detection + Twilio SMS trigger for welcome sequence", "create WordPress table for SMS responses", "build SMS-unsubscribe handler that updates both WordPress and MailerLite"]}""",
    """{"triggers": ["new enterprise signup", "existing 'Enterprise Leads' list already exists in MailerLite"], "actions": ["create new MailerLite list for enterprise signups", "notify sales via Slack", "also notify sales via email for the same event"], "action_items": ["create enterprise signups list", "Slack webhook", "duplicate email notification to sales"]}""",
]

for i, test_input in enumerate(test_inputs, start=1):
    print(f"\n{'='*80}\nTEST INPUT {i}\n{'='*80}")

    orig_resp = client.messages.create(
        model="claude-opus-4-8", max_tokens=500, system=build_planning_original_prompt,
        messages=[{"role": "user", "content": f"Extracted requirements:\n\n{test_input}\n\nProduce the build plan."}],
    )
    comp_resp = client.messages.create(
        model="claude-opus-4-8", max_tokens=500, system=build_planning_compressed_prompt,
        messages=[{"role": "user", "content": f"Extracted requirements:\n\n{test_input}\n\nProduce the build plan."}],
    )

    print(f"--- ORIGINAL (input_tokens={orig_resp.usage.input_tokens}, output_tokens={orig_resp.usage.output_tokens}) ---")
    print(orig_resp.content[0].text[:300])
    print(f"\n--- COMPRESSED (input_tokens={comp_resp.usage.input_tokens}, output_tokens={comp_resp.usage.output_tokens}) ---")
    print(comp_resp.content[0].text[:300])

original: 4010 chars (~1002 tokens)
compressed: 2687 chars (~671 tokens)

TEST INPUT 1
--- ORIGINAL (input_tokens=1494, output_tokens=500) ---
{
  "build_steps": [
    {
      "step_number": 1,
      "platform": "mailerlite",
      "description": "Create or reuse a subscribers list to receive new signups and drive automation sequences",
      "depends_on": [],
      "reuses_existing": "Subscribers list"
    },
    {
      "step_number": 2,

--- COMPRESSED (input_tokens=1114, output_tokens=500) ---
```json
{
  "build_steps": [
    {"step_number": 1, "platform": "mailerlite", "description": "Create or reuse a 'Subscribers' list to hold new signups; add a segment for enterprise signups based on a plan/type field", "depends_on": [], "reuses_existing": "Subscribers list (default) if present"},
   

TEST INPUT 2
--- ORIGINAL (input_tokens=1636, output_tokens=500) ---
{
  "build_steps": [
    {
      "step_number": 1,
      "platform": "wordpress",
      "description": "Ensure WordPress subsc

## Step 7 — Technique 3: Model downgrade

From your Lesson 3 model comparison, identify one task where a cheaper model produced acceptable output. Update the cost table entry with the cheaper model and recalculate.

**Quality impact:** Reference your Lesson 3 comparison. Document: "Haiku scored X vs Sonnet's Y on [criteria]. Acceptable because [reason]."

In [ ]:
# Downgrade candidate: Payload Generation (currently claude-sonnet)
original_model  = "claude-sonnet"
downgrade_model = "gemini-flash-lite"

task_input_tokens  = 1800
task_output_tokens = 900
requests_per_day   = 8

original_monthly   = estimate_monthly(original_model, task_input_tokens, task_output_tokens, requests_per_day)
downgraded_monthly = estimate_monthly(downgrade_model, task_input_tokens, task_output_tokens, requests_per_day)
savings_pct = (1 - downgraded_monthly / original_monthly) * 100

print(f"Original ({original_model}):    ${original_monthly:.4f}/month")
print(f"Downgraded ({downgrade_model}): ${downgraded_monthly:.4f}/month")
print(f"Savings: {savings_pct:.1f}%")

Original (claude-sonnet):    $3.0240/month
Downgraded (gemini-flash-lite): $0.6696/month
Savings: 77.9%


## Step 8 — Technique 4: Output length control

Add `max_tokens` and explicit conciseness instructions to one prompt. Measure the output token reduction.

**Quality impact:** If the output is truncated mid-sentence, `max_tokens` is too low. Test on 3 inputs.

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def get_text(response) -> str:
    """Extract the text content from a response, skipping ThinkingBlock/other block types."""
    for block in response.content:
        if block.type == "text":
            return block.text
    return ""  # no text block found (shouldn't normally happen)


verification_system_prompt = "You write plain-language verification summaries of completed automation builds for non-technical stakeholders."

build_logs = [
    "Build log: welcome email (MailerLite), day-3 follow-up (MailerLite), Slack webhook "
    "(enterprise only), CRM contact logging (Make). All 4 steps succeeded. Reused: existing "
    "'New Signups 2026' list instead of creating duplicate.",

    "Build log: MailerLite unsubscribe -> WordPress subscribers table sync (webhook), Twilio "
    "SMS follow-up for non-responders after 3 days, WordPress table for SMS responses, "
    "SMS-unsubscribe handler updating both WordPress and MailerLite. All 4 steps succeeded. "
    "No existing resources reused; all new.",

    "Build log: Enterprise Signups segment created on existing 'Enterprise Leads' list "
    "(reused, not duplicated). Make scenario adds contact to segment, then Router branches "
    "to Slack notification and email notification for sales. All steps succeeded. One "
    "advisory conflict noted and resolved: avoided creating a duplicate MailerLite list.",
]

for i, build_log in enumerate(build_logs, start=1):
    print(f"\n{'='*80}\nBUILD LOG {i}\n{'='*80}\n{build_log}\n")

    r_long = client.messages.create(
        model="claude-sonnet-5",
        max_tokens=1024,
        system=verification_system_prompt,
        messages=[{"role": "user", "content": f"Write a verification summary for this build:\n\n{build_log}"}],
    )

    r_short = client.messages.create(
        model="claude-sonnet-5",
        max_tokens=150,
        system=verification_system_prompt,
        messages=[{"role": "user", "content": f"Write a verification summary for this build:\n\n{build_log}\n\nRespond in under 80 words."}],
    )

    long_out = r_long.usage.output_tokens
    short_out = r_short.usage.output_tokens
    reduction_abs = long_out - short_out
    reduction_pct = (reduction_abs / long_out) * 100 if long_out else 0

    print(f"Long output_tokens:  {long_out}")
    print(f"Short output_tokens: {short_out}")
    print(f"Reduction: {reduction_abs} tokens ({reduction_pct:.1f}%)")
    print(f"\nr_short text:\n{get_text(r_short)}")

    print(f"\nr_short stop_reason: {r_short.stop_reason}  "
          f"({'OK - finished naturally' if r_short.stop_reason == 'end_turn' else 'WARNING - may be truncated, max_tokens too low'})")


BUILD LOG 1
Build log: welcome email (MailerLite), day-3 follow-up (MailerLite), Slack webhook (enterprise only), CRM contact logging (Make). All 4 steps succeeded. Reused: existing 'New Signups 2026' list instead of creating duplicate.

Long output_tokens:  428
Short output_tokens: 150
Reduction: 278 tokens (65.0%)

r_short text:
**Build Verification Summary**

All 4 automation steps completed successfully:

1. ✅ Welcome email sent via MailerLite
2. ✅ Day-3 follow-up email scheduled via MailerLite
3. ✅ Slack notification triggered (enterprise-only feature)
4. ✅ New contact logged in CRM via Make

**Note:** The system reused your existing "

r_short stop_reason: max_tokens  (WARNING - may be truncated, max_tokens too low)

BUILD LOG 2
Build log: MailerLite unsubscribe -> WordPress subscribers table sync (webhook), Twilio SMS follow-up for non-responders after 3 days, WordPress table for SMS responses, SMS-unsubscribe handler updating both WordPress and MailerLite. All 4 steps succeede

## Step 9 — Technique 5: Batching (if applicable)

If any capstone task is NOT latency-sensitive (nightly reports, bulk processing, dataset labeling), calculate the batch API discount. Anthropic's Message Batches API processes requests asynchronously at a lower per-token cost.

**Quality impact:** None — same model, same output. The tradeoff is latency, not quality.

In [ ]:
# VOLATILE: verify batch discount from Anthropic docs
# Confirmed as of this check: https://platform.claude.com/docs/en/build-with-claude/batch-processing
# "All usage is charged at 50% of the standard API prices."
BATCH_DISCOUNT = 0.50

# Applied to: Verification Summary — the one capstone task with no downstream dependency
# on real-time completion. Nothing in the build pipeline blocks on this step finishing
# immediately; it's a plain-language recap of an already-completed build, read later by
# a stakeholder. Contrast with Error Recovery (must unblock a stuck build now) and
# Payload Generation (blocks the next build step) — both latency-sensitive, not batch candidates.
model = "gemini-flash-lite"
in_tokens, out_tokens = 84, 95
requests_per_day = 2

standard_monthly = estimate_monthly(model, in_tokens, out_tokens, requests_per_day)
batch_monthly = standard_monthly * BATCH_DISCOUNT
savings_abs = standard_monthly - batch_monthly

print(f"Standard monthly: ${standard_monthly:.4f}")
print(f"Batch monthly:    ${batch_monthly:.4f}")
print(f"Absolute savings: ${savings_abs:.4f}/month")

Standard monthly: $0.0158
Batch monthly:    $0.0079
Absolute savings: $0.0079/month


## Step 10 — Before/after comparison table

Produce a comparison table showing the original vs optimized monthly cost for each task. Update the `optimized_tasks` list with the optimizations you actually applied.

In [ ]:
PRICING = {
    "claude-haiku":  {"input_per_1m": 1.00,  "output_per_1m": 5.00},
    "claude-sonnet": {"input_per_1m": 2.00,  "output_per_1m": 10.00},
    "claude-opus":   {"input_per_1m": 5.00,  "output_per_1m": 25.00},
    "claude-fable":  {"input_per_1m": 10.00, "output_per_1m": 50.00},
    "gemini-flash-lite": {"input_per_1m": 0.30, "output_per_1m": 2.50},
    "mistral-small":     {"input_per_1m": 0.15, "output_per_1m": 0.60},
}

def calculate_cost(model, input_tokens, output_tokens):
    rates = PRICING[model]
    return (input_tokens * rates["input_per_1m"] / 1_000_000
            + output_tokens * rates["output_per_1m"] / 1_000_000)

def estimate_monthly(model, input_tokens, output_tokens, requests_per_day):
    return calculate_cost(model, input_tokens, output_tokens) * requests_per_day * 30

BATCH_DISCOUNT = 0.50

original_tasks = [
    {"name": "Extraction",           "model": "claude-sonnet",     "input_tokens": 2500, "output_tokens": 800,  "requests_per_day": 2},
    {"name": "Build Planning",       "model": "claude-opus",       "input_tokens": 3000, "output_tokens": 1500, "requests_per_day": 2},
    {"name": "Payload Generation",   "model": "claude-sonnet",     "input_tokens": 1800, "output_tokens": 900,  "requests_per_day": 8},
    {"name": "Error Recovery",       "model": "gemini-flash-lite", "input_tokens": 500,  "output_tokens": 200,  "requests_per_day": 3},
    {"name": "Verification Summary", "model": "gemini-flash-lite", "input_tokens": 1000, "output_tokens": 300,  "requests_per_day": 2},
]

optimized_tasks = [
    {"name": "Extraction",           "model": "claude-sonnet",     "input_tokens": 2500, "output_tokens": 800,
     "requests_per_day": 2, "optimizations": "none (caching tested; system prompt below Sonnet's 1024-token cache floor)"},
    {"name": "Build Planning",       "model": "claude-opus",       "input_tokens": 2013, "output_tokens": 1500,
     "requests_per_day": 2, "optimizations": "prompt compression (-32.9%, verified equivalent output on 3 tests)"},
    {"name": "Payload Generation",   "model": "gemini-flash-lite", "input_tokens": 1800, "output_tokens": 900,
     "requests_per_day": 8, "optimizations": "model downgrade (reasoned from Exp 2/4, not yet measured)"},
    {"name": "Error Recovery",       "model": "gemini-flash-lite", "input_tokens": 66,   "output_tokens": 78,
     "requests_per_day": 3, "optimizations": "right-sized after measuring a real sample prompt"},
    {"name": "Verification Summary", "model": "gemini-flash-lite", "input_tokens": 84,   "output_tokens": 95,
     "requests_per_day": 2, "optimizations": "right-sized + Batch API (50% discount)"},
]

rows = []
total_orig, total_opt = 0.0, 0.0

for orig, opt in zip(original_tasks, optimized_tasks):
    m_orig = estimate_monthly(orig["model"], orig["input_tokens"], orig["output_tokens"], orig["requests_per_day"])
    m_opt = estimate_monthly(opt["model"], opt["input_tokens"], opt["output_tokens"], opt["requests_per_day"])
    if "Batch API" in opt["optimizations"]:
        m_opt *= BATCH_DISCOUNT
    savings_pct = (1 - m_opt / m_orig) * 100 if m_orig else 0
    total_orig += m_orig
    total_opt += m_opt
    rows.append((orig["name"], orig["model"], m_orig, opt["model"], opt["optimizations"], m_opt, savings_pct))

header = f"{'Task':<22}{'Orig Model':<17} {'Orig $/mo':>10}  {'Opt Model':<19} {'Optimizations':<58}{'Opt $/mo':>10}{'Savings':>9}"
print(header)
print("-" * len(header))
for name, orig_model, m_orig, opt_model, opts, m_opt, pct in rows:
    opts_display = (opts[:55] + "...") if len(opts) > 58 else opts
    print(f"{name:<22}{orig_model:<17} {'$'+f'{m_orig:.4f}':>10}  {opt_model:<19} {opts_display:<58}{'$'+f'{m_opt:.4f}':>10}{pct:>8.1f}%")

print("-" * len(header))
total_savings_pct = (1 - total_opt / total_orig) * 100 if total_orig else 0
print(f"{'TOTAL':<22}{'':<17} {'$'+f'{total_orig:.4f}':>10}  {'':<19} {'':<58}{'$'+f'{total_opt:.4f}':>10}{total_savings_pct:>8.1f}%")

Task                  Orig Model         Orig $/mo  Opt Model           Optimizations                                               Opt $/mo  Savings
-----------------------------------------------------------------------------------------------------------------------------------------------------
Extraction            claude-sonnet        $0.7800  claude-sonnet       none (caching tested; system prompt below Sonnet's 1024...   $0.7800     0.0%
Build Planning        claude-opus          $3.1500  claude-opus         prompt compression (-32.9%, verified equivalent output ...   $2.8539     9.4%
Payload Generation    claude-sonnet        $3.0240  gemini-flash-lite   model downgrade (reasoned from Exp 2/4, not yet measured)    $0.6696    77.9%
Error Recovery        gemini-flash-lite    $0.0585  gemini-flash-lite   right-sized after measuring a real sample prompt             $0.0193    67.0%
Verification Summary  gemini-flash-lite    $0.0630  gemini-flash-lite   right-sized + Batch API (50%

---

## Part 3 — Cost Monitoring (~10 min)

## Step 11 — Cost logging function

After each API call, append a row to `cost_log.csv` with: timestamp, model, task name, input tokens, output tokens, cost in USD. This is the seed for M6's cost monitoring dashboard.

In [ ]:
import csv
import os
from datetime import datetime

LOG_FILE = "cost_log.csv"
LOG_HEADER = ["timestamp", "model", "task_name", "input_tokens", "output_tokens", "cost_usd"]


def log_cost(task_name: str, model: str, usage, pricing_key: str) -> None:
    """Append a cost entry to cost_log.csv."""
    rates = PRICING[pricing_key]

    input_cost = usage.input_tokens * (rates["input_per_1m"] / 1_000_000)
    output_cost = usage.output_tokens * (rates["output_per_1m"] / 1_000_000)
    total = input_cost + output_cost

    file_exists = os.path.isfile(LOG_FILE)

    with open(LOG_FILE, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(LOG_HEADER)
        writer.writerow([
            datetime.now().isoformat(),
            model,
            task_name,
            usage.input_tokens,
            usage.output_tokens,
            f"{total:.6f}",
        ])

print("log_cost() defined.")

log_cost() defined.


## Step 12 — Run 10 logged API calls

Run a mix of tasks and call `log_cost()` after each. Verify the CSV contains 10 rows.

In [ ]:
import os
import csv

MODEL = "claude-sonnet-5"
MODEL_FAMILY = "claude-sonnet"

# Remove old log if re-running
if os.path.exists(LOG_FILE):
    os.remove(LOG_FILE)

sample_tasks = [
    ("classification", "Classify: My order is late."),
    ("classification", "Classify: I was charged twice."),
    ("classification", "Classify: The product arrived broken."),
    ("summarization",  "Summarize in one sentence: Machine learning models learn patterns from data to make predictions on new inputs."),
    ("summarization",  "Summarize in one sentence: Transformers use self-attention to process all input tokens in parallel."),
    ("summarization",  "Summarize in one sentence: Retrieval-augmented generation combines search with language models."),
    ("generation",     "Write a one-paragraph product description for wireless noise-cancelling headphones."),
    ("generation",     "Write a one-paragraph product description for a portable espresso maker."),
    ("generation",     "Write a one-paragraph product description for a smart water bottle."),
    ("classification", "Classify: I can't reset my password."),
]

for task_name, prompt in sample_tasks:
    resp = client.messages.create(
        model=MODEL,
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}],
    )
    log_cost(task_name, MODEL, resp.usage, MODEL_FAMILY)

with open(LOG_FILE) as f:
    reader = csv.reader(f)
    rows = list(reader)

header, data_rows = rows[0], rows[1:]
print(f"Row count (excluding header): {len(data_rows)}")
print(f"\nFirst 3 rows:")
for row in data_rows[:3]:
    print(row)

Row count (excluding header): 10

First 3 rows:
['2026-08-24T08:46:19.736414', 'claude-sonnet-5', 'classification', '15', '200', '0.002030']
['2026-08-24T08:46:23.292358', 'claude-sonnet-5', 'classification', '17', '200', '0.002034']
['2026-08-24T08:46:26.782058', 'claude-sonnet-5', 'classification', '18', '200', '0.002036']


## Step 13 — Cost summary

Read `cost_log.csv` and print: total cost, average cost per request, cost by model, cost by task.

In [ ]:
import csv
from collections import defaultdict

costs_by_model = defaultdict(float)
costs_by_task = defaultdict(float)
total_cost = 0.0
count = 0

with open(LOG_FILE) as f:
    reader = csv.DictReader(f)
    for row in reader:
        cost = float(row["cost_usd"])
        total_cost += cost
        count += 1
        costs_by_model[row["model"]] += cost
        costs_by_task[row["task_name"]] += cost

avg_cost = total_cost / count if count else 0
print(f"Total cost:        ${total_cost:.6f}")
print(f"Average per call:   ${avg_cost:.6f}")
print(f"Total calls:        {count}")

print("\nCost by model:")
for model, cost in sorted(costs_by_model.items(), key=lambda x: -x[1]):
    print(f"  {model:<20} ${cost:.6f}")

print("\nCost by task:")
for task, cost in sorted(costs_by_task.items(), key=lambda x: -x[1]):
    print(f"  {task:<20} ${cost:.6f}")

Total cost:        $0.016258
Average per call:   $0.001626
Total calls:        10

Cost by model:
  claude-sonnet-5      $0.016258

Cost by task:
  classification       $0.008136
  generation           $0.006170
  summarization        $0.001952


---

## Your turn

Choose **one** of these open-ended challenges:

1. **Cost alert:** Add a check to `log_cost()` that prints a warning if the cumulative daily cost exceeds a configurable threshold. Run enough calls to trigger it. This is the seed for production cost monitoring.

2. **Cache break-even:** Vary the length of your cached prefix — try 500, 1,000, and 2,000 tokens. At what point does caching activate? What's the break-even point where the cache-write cost pays for itself?

3. **Cost projection:** Given your current daily cost and a weekly growth rate of 10%, project monthly costs for the next 6 months. At what month does cost become unsustainable without further optimization?

Pick the one that's most relevant to your capstone. Show your work and explain your reasoning.

In [ ]:
# Re-run this exact cell to make sure the ALERT version is active
import csv, os
from datetime import datetime, date

print(f"[debug] Defining log_cost() now. Working directory: {os.getcwd()}")

LOG_FILE = "cost_log.csv"
LOG_HEADER = ["timestamp", "model", "task_name", "input_tokens", "output_tokens", "cost_usd"]
DAILY_COST_THRESHOLD = 0.001
_alert_state = {"date": None, "already_alerted": False}

print(f"[debug] LOG_FILE resolves to: {os.path.abspath(LOG_FILE)}")
print(f"[debug] LOG_FILE exists already? {os.path.isfile(LOG_FILE)}")

def _today_total_cost():
    if not os.path.isfile(LOG_FILE):
        return 0.0
    today_str = date.today().isoformat()
    total = 0.0
    with open(LOG_FILE) as f:
        for row in csv.DictReader(f):
            if row["timestamp"].startswith(today_str):
                total += float(row["cost_usd"])
    return total

def log_cost(task_name, model, usage, pricing_key, daily_threshold=DAILY_COST_THRESHOLD):
    print(f"[debug] log_cost() CALLED — task={task_name!r} model={model!r} pricing_key={pricing_key!r} "
          f"in={usage.input_tokens} out={usage.output_tokens}")

    rates = PRICING[pricing_key]
    total = usage.input_tokens * (rates["input_per_1m"]/1_000_000) + usage.output_tokens * (rates["output_per_1m"]/1_000_000)
    print(f"[debug] computed cost for this call: ${total:.6f}")

    file_exists = os.path.isfile(LOG_FILE)
    print(f"[debug] LOG_FILE exists before write? {file_exists} -> "
          f"{'appending row' if file_exists else 'creating file + header row'}")

    with open(LOG_FILE, "a", newline="") as f:
        w = csv.writer(f)
        if not file_exists:
            w.writerow(LOG_HEADER)
        w.writerow([datetime.now().isoformat(), model, task_name, usage.input_tokens, usage.output_tokens, f"{total:.6f}"])

    print(f"[debug] row written. LOG_FILE now exists? {os.path.isfile(LOG_FILE)} "
          f"size={os.path.getsize(LOG_FILE) if os.path.isfile(LOG_FILE) else 0} bytes")

    today = date.today().isoformat()
    if _alert_state["date"] != today:
        _alert_state["date"] = today
        _alert_state["already_alerted"] = False

    cumulative = _today_total_cost() + total
    print(f"[debug] cumulative so far: ${cumulative:.6f} (threshold ${daily_threshold})")

    if cumulative > daily_threshold and not _alert_state["already_alerted"]:
        print(f"⚠️  COST ALERT: ${cumulative:.4f} exceeded ${daily_threshold}/day (triggered by {task_name} on {model})")
        _alert_state["already_alerted"] = True
    else:
        print(f"[debug] no alert fired this call (already_alerted={_alert_state['already_alerted']})")

print("log_cost() (debug build) defined.")


# --- Self-test: runs automatically, no real API call needed ---
class FakeUsage:
    def __init__(self, i, o): self.input_tokens, self.output_tokens = i, o

print("\n[debug] Running a self-test call now...")
log_cost("selftest", "claude-sonnet-5", FakeUsage(50, 20), "claude-sonnet")
print("[debug] Self-test complete. If you saw '[debug] log_cost() CALLED' above, the function works.")

[debug] Defining log_cost() now. Working directory: /content
[debug] LOG_FILE resolves to: /content/cost_log.csv
[debug] LOG_FILE exists already? True
log_cost() (debug build) defined.

[debug] Running a self-test call now...
[debug] log_cost() CALLED — task='selftest' model='claude-sonnet-5' pricing_key='claude-sonnet' in=50 out=20
[debug] computed cost for this call: $0.000300
[debug] LOG_FILE exists before write? True -> appending row
[debug] row written. LOG_FILE now exists? True size=863 bytes
[debug] cumulative so far: $0.016858 (threshold $0.001)
⚠️  COST ALERT: $0.0169 exceeded $0.001/day (triggered by selftest on claude-sonnet-5)
[debug] Self-test complete. If you saw '[debug] log_cost() CALLED' above, the function works.


---

## Quality checklist

Evaluate your own work before marking the exercise complete:

- [x] Cost calculator correctly computes per-request and monthly costs (verify with manual calculation on one example)
- [partial ] Capstone cost table covers all identified AI tasks with volume estimates grounded in product requirements
- [x] At least 3 optimization techniques applied with documented cost reduction
- [ ] Each optimization includes a quality impact assessment — not just "it's cheaper" but "quality held / degraded by X because Y"
- [x] Before/after cost comparison table produced with percentage savings
- [x] Cost monitoring CSV logs correctly with all required fields (timestamp, model, task, input_tokens, output_tokens, cost_usd)
- [x] Summary script reads the CSV and produces per-model and per-task breakdowns

## What I learned

Write 3–6 bullets in your own words, covering:

- Where this connects to your capstone


- What surprised me: after building out the full cost-alert and logging system, I couldn't find the file, it turned out to be sitting in Colab's ephemeral local storage the whole time.
- One thing I'd do differently in production: I'd persist the cost log and alert state somewhere durable instead of Colab's local runtime storage, since I already lost track of cost_log.csv once and the alert's "already fired today" flag silently resets on every kernel restart.
- Where this connects to my capstone: the Payload Generation model downgrade is my single biggest projected cost lever at 77.9% savings.

_This cell is required — `check-notebook.py` enforces it. It's also what makes this notebook a portfolio artifact when you publish to GitHub._